#### Struct Example

In [0]:
#step 1: import necessary modules
from pyspark.sql import Row
from pyspark.sql.types import StructType,StructField, IntegerType,StringType

In [0]:
#step 2 :create sample data

data = [(1, "Alice",
         Row(Street ="123 MaSn St",
             City= "New York",
             State ="NY", 
             Zip ="10001")),
            (2, "Bob",
         Row(Street ="456 Buffalo St",
             City= "New Jersey",
             State ="NJ", 
             Zip ="50005")),
            (3, "Andrew",
         Row(Street ="455 Osborne St",
             City= "Washington DC",
             State ="Washington", 
             Zip ="71239"))
            ]

In [0]:
for item in data:
    print(item[2][1])

In [0]:
####Step 3: Define the Schema with STRUCT

In [0]:

address_schema = StructType(
                   [StructField("Street", StringType()),
                    StructField("City", StringType()),
                    StructField("State", StringType()),
                    StructField("Zip", StringType())])

data_schema = StructType(
                [StructField("id", IntegerType(),True),
               StructField("name", StringType(),True),
               StructField("address",address_schema,True)])

            
              

##### Step 4 : Create a Dataframe and display it

In [0]:
df_struct = spark.createDataFrame(data =data,schema= data_schema)

In [0]:
df_struct.show(truncate=False)

In [0]:
df_struct.printSchema()

In [0]:
df_struct.display()

In [0]:
from pyspark.sql.functions import col

df_access_struct = df_struct.select(col('name'), col('address.City'),col('address.State'))

df_access_struct.show(truncate=False)

In [0]:
##### Step 5: Accessing the Struct Fields

In [0]:
address_extract=df_struct.select('address')

In [0]:
address_extract.display()

In [0]:
address_extract.select('address.Street').display()

#### Array Example

In [0]:
from pyspark.sql import Row
array_data =[(
1,'Alice',
[Row(skill ='Python', years_of_exp = 6),
 Row(skill ='SQL', years_of_exp = 8),
 Row(skill ='Databricks', years_of_exp = 5)
 ]),
(2,'Bob',[Row(skill ='SQL',years_of_exp =5)]),
(
3,'Focus',
[Row(skill ='Python', years_of_exp = 4),
 Row(skill ='SQL', years_of_exp = 6)
 ])]


In [0]:
for object in array_data:
    print(object)

 ###### so we see variable nature of array in the row objects through skills info but structure is the same. Alice has 3 skills, Bob 1 skill and Focus 2 skills

#### Step 2 : Define the Schema with Array


In [0]:
from pyspark.sql.types import ArrayType


In [0]:

skills_col_schema = ArrayType(
                     StructType([
                        StructField("skill" ,StringType()),
                        StructField("years_of_exp",IntegerType())
                            ]));
                            
person_skills_schema= StructType([
                        StructField("id",IntegerType(),True),
                        StructField("name", StringType(),True),
                        StructField("skills", skills_col_schema,True)
        ])

In [0]:
skills_df =spark.createDataFrame(data = array_data, schema = person_skills_schema)

In [0]:
skills_df.printSchema()

In [0]:
skills_df.display()

In [0]:
access_skills_df = skills_df.select(col("skills"))

access_skills_df.display()
                                    
                                    

In [0]:
from pyspark.sql.functions import get

access_skills_df_1st_element = skills_df.select(
                    get(col("skills")["skill"],2).alias("3rd_skill"))
                                    #using get function to tolerate array rows with out of bounds and return NULL instead

access_skills_df_1st_element.display()

In [0]:
skill_info = skills_df.select(col("name"), get(col("skills"),0).alias('1st_skill'))

In [0]:
skill_info.display()

In [0]:
# access the nested field
access_nested_skill_info = skills_df.select(col("name"), get(col("skills"),0)['skill'].alias('1st_skill_name'))


In [0]:
access_nested_skill_info.display()

In [0]:
skills_df.display()

In [0]:
from pyspark.sql.functions import explode
explode_skills_array = skills_df.select(col("name"),
                                        explode(col("skills")).alias("skill_detail"))

In [0]:
explode_skills_array.display()

In [0]:
explode_skills_array.printSchema()

In [0]:
skills_df.printSchema()

In [0]:
flatten_table = explode_skills_array.select(
                                        col("name"),
                                        col("skill_detail").skill.alias("skill"),
                                        col("skill_detail").years_of_exp.alias("years_of_experience"))

In [0]:
flatten_table.display()

#### MAP Example

In [0]:
map_data =[
 (1, "Alice", {"phone": "123-4567","email" :"alice@email.com"}),
 (2, "Bob", {"whatsapp": "+1-555-0100", "telegram": "@bob"}),
(3, "Charlie ", {"email": "charlie@email.com", "linkedin": "linkedin.com/charlie", "slack": "@charlie"})]


In [0]:

#length_map_raw_data = print(len(map_data))

In [0]:
from pyspark.sql.types import MapType,StructType, StructField, IntegerType, StringType

map_data_schema_col = MapType(StringType(), StringType());

data_schema = StructType([
            StructField("id",IntegerType(),True),
            StructField("name",StringType(),True),
            StructField("contact_info",map_data_schema_col,True)
            ])


            

In [0]:
map_data_df = spark.createDataFrame(
            data= map_data,schema = data_schema)

map_data_df.display()

In [0]:
map_data_df_access = map_data_df.select(col("contact_info")["phone"])

In [0]:
map_data_df_info_extract = map_data_df.select(
    col("name"),
    col("contact_info")["phone"].alias("phone"),
    col("contact_info")["email"].alias("email"),
    col("contact_info")["whatsapp"].alias("whatsapp")
    )
                                              

In [0]:
map_data_df_info_extract.display()

In [0]:
from pyspark.sql.functions import explode

explode_map_contact_data = map_data_df.select(col("name"), explode("contact_info").alias(
    "contact_type", "contact_value")
)

In [0]:
explode_map_contact_data.display()

In [0]:
map_data_df.display()

In [0]:
from pyspark.sql.functions import map_keys, map_values

map_keys_and_values =map_data_df.select(
            col("name"),
            map_keys(col("contact_info")).alias("contact_type") , 
            map_values(col("contact_info")).alias("contact_value")

)

In [0]:
map_keys_and_values.display()